# Feature Selection Demo: Breast Cancer Dataset

## Load Data

In [101]:

from sklearn.datasets import load_breast_cancer
import pandas as pd

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

print(X.shape)
X.head()


(569, 30)


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


## Train/Test Split

In [102]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


In [103]:
def evaluate_model(X_train, X_test, y_train, y_test):
    model = LogisticRegression(max_iter=5000)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    return accuracy_score(y_test, pred)

## Baseline Model

In [104]:
model = LogisticRegression(max_iter=5000)
model.fit(X_train, y_train)

pred = model.predict(X_test)
print("Baseline accuracy:", accuracy_score(y_test, pred))


Baseline accuracy: 0.9766081871345029


## Variance Threshold

In [105]:

from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0.01)
X_train_var = selector.fit_transform(X_train)
X_test_var = selector.transform(X_test)

print("Original:", X_train.shape[1], "Selected:", X_train_var.shape[1])


Original: 30 Selected: 14


In [106]:
acc_var = evaluate_model(X_train_var, X_test_var, y_train, y_test)
print("Variance Threshold accuracy:", acc_var)

Variance Threshold accuracy: 0.9766081871345029


## Mutual Information (SelectKBest)

In [107]:

from sklearn.feature_selection import SelectKBest, mutual_info_classif

selector = SelectKBest(mutual_info_classif, k=10)
X_train_mi = selector.fit_transform(X_train, y_train)
X_test_mi = selector.transform(X_test)

selected_features = X.columns[selector.get_support()]
print("Selected:", X_train_mi.shape[1], " features")
print("Selected features:", list(selected_features))


Selected: 10  features
Selected features: ['mean radius', 'mean perimeter', 'mean area', 'mean concavity', 'mean concave points', 'area error', 'worst radius', 'worst perimeter', 'worst area', 'worst concave points']


In [108]:
acc_mi= evaluate_model(X_train_mi, X_test_mi, y_train, y_test)
print("MI Threshold accuracy:", acc_mi)

MI Threshold accuracy: 0.9707602339181286


## Mutual Information Feature Selection (MIFS)

In [109]:
from sklearn.feature_selection import mutual_info_regression

def mifs(X, y, k, beta=0.5):
    n_features = X.shape[1]
    
    mi = mutual_info_classif(X, y)
    selected = [np.argmax(mi)]
    
    while len(selected) < k:
        scores = []
        
        for j in range(n_features):
            if j in selected:
                continue
            
            redundancy = np.mean([
            mutual_info_regression(X[:, [j]], X[:, s])[0]
            for s in selected
            ])
            
            score = mi[j] - beta * redundancy
            scores.append((score, j))
        
        selected.append(max(scores)[1])
    
    return selected

In [110]:
selected_idx = mifs(X_train.values, y_train, k=10)

X_train_mifs = X_train.iloc[:, selected_idx]
X_test_mifs = X_test.iloc[:, selected_idx]

# number of selected features
print("Number of selected features:", len(selected_idx))

# feature names
selected_features = X_train.columns[selected_idx]
print("Selected features:", list(selected_features))

acc_mifs = evaluate_model(X_train_mifs, X_test_mifs, y_train, y_test)
print("MIFS accuracy:", acc_mifs)

Number of selected features: 10
Selected features: ['worst perimeter', 'worst concave points', 'worst texture', 'area error', 'mean concave points', 'worst concavity', 'worst area', 'perimeter error', 'mean concavity', 'mean perimeter']
MIFS accuracy: 0.9707602339181286


## Relief, An Approximation

In [116]:
from sklearn.neighbors import NearestNeighbors

def relief(X, y, n_samples=100):
    n_features = X.shape[1]
    weights = np.zeros(n_features)
    
    nbrs = NearestNeighbors(n_neighbors=2).fit(X)
    
    for _ in range(n_samples):
        i = np.random.randint(0, X.shape[0])
        xi = X[i]
        yi = y[i]
        
        # find neighbors
        distances, indices = nbrs.kneighbors([xi])
        
        hit = None
        miss = None
        
        for idx in indices[0][1:]:
            if y[idx] == yi and hit is None:
                hit = X[idx]
            elif y[idx] != yi and miss is None:
                miss = X[idx]
        
        if hit is None or miss is None:
            continue
        
        weights -= np.abs(xi - hit)
        weights += np.abs(xi - miss)
    
    return weights

In [117]:
# Step 1: run Relief to get feature weights
weights = relief(X_train.values, y_train, n_samples=200)

# Step 2: select top-k features
k = 10
top_idx = np.argsort(weights)[-k:]

# Step 3: print selected features
selected_features = X_train.columns[top_idx]
print("Number of selected features:", len(selected_features))
print("Selected features:", list(selected_features))

# Step 4: reduce dataset
X_train_relief = X_train.iloc[:, top_idx]
X_test_relief = X_test.iloc[:, top_idx]

# Step 5: train model and evaluate
acc_relief = evaluate_model(X_train_relief, X_test_relief, y_train, y_test)

print("Relief accuracy:", acc_relief)

Number of selected features: 10
Selected features: ['mean symmetry', 'mean concave points', 'mean concavity', 'mean compactness', 'mean smoothness', 'mean area', 'mean perimeter', 'mean texture', 'worst symmetry', 'worst fractal dimension']
Relief accuracy: 0.9415204678362573


## Recursive Feature Elimination (RFE) (Wrapper Method)

In [118]:

from sklearn.feature_selection import RFE

model = LogisticRegression(max_iter=5000)
selector = RFE(model, n_features_to_select=14)
X_train_rfe = selector.fit_transform(X_train, y_train)
X_test_rfe = selector.transform(X_test)

selected_features = X.columns[selector.get_support()]
print("Selected features:", list(selected_features))


Selected features: ['mean radius', 'mean texture', 'mean compactness', 'mean concavity', 'mean concave points', 'mean symmetry', 'texture error', 'perimeter error', 'worst texture', 'worst smoothness', 'worst compactness', 'worst concavity', 'worst concave points', 'worst symmetry']


In [119]:
acc_rfe = evaluate_model(X_train_rfe, X_test_rfe, y_train, y_test)
print("REF Threshold accuracy:", acc_rfe)

REF Threshold accuracy: 0.9649122807017544


## L1 Logistic Regression (Embedded)

In [120]:
weights = relief(X_train.values, y_train, n_samples=200)

# select top 10 features
top_idx = np.argsort(weights)[-10:]

X_train_relief = X_train.iloc[:, top_idx]
X_test_relief = X_test.iloc[:, top_idx]

acc_relief = evaluate_model(X_train_relief, X_test_relief, y_train, y_test)
print("Relief accuracy:", acc_relief)

Relief accuracy: 0.9415204678362573


In [121]:

# Step 1: Train L1 logistic regression (for feature selection)
model_l1 = LogisticRegression(
    penalty='l1',            # remove penalty usage
    solver='liblinear',           # saga supports elastic net
    C=0.001,            # 1.0 = pure L1, 0.001 is strong C
    max_iter=5000
)
model_l1.fit(X_train, y_train)

# Step 2: Get indices of selected features (non-zero coefficients)
selected_idx = np.where(model_l1.coef_[0] != 0)[0]

print("Number of selected features:", len(selected_idx))
print("Selected features:", X.columns[selected_idx])

# Step 3: Reduce dataset
X_train_l1 = X_train.iloc[:, selected_idx]
X_test_l1 = X_test.iloc[:, selected_idx]

# Step 4: Train a standard logistic regression on selected features
model = LogisticRegression(max_iter=5000)
model.fit(X_train_l1, y_train)

# Step 5: Evaluate
pred = model.predict(X_test_l1)
acc_l1 = accuracy_score(y_test, pred)

print("L1-selected features accuracy:", acc_l1)

Number of selected features: 3
Selected features: Index(['mean perimeter', 'mean area', 'worst area'], dtype='str')
L1-selected features accuracy: 0.9766081871345029
